# Chapter 7 - Lab 4: Adding an evaluator-optimizer to the pipeline (customer-care edition)

**Goal**: extend Lab 1 with a fifth agent, the **EvaluatorAgent**, that reviews the Supervisor's account-health
assessment and sends it back for revision if it is incomplete. This is the contact-centre **QA gate**, built into
the workflow.

The evaluator does **not** ask an LLM "is this good?". It runs **deterministic checks** on the shared state:
were all 4 service-quality and all 4 relationship-risk metrics scored, and did the Supervisor produce a verdict?

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%207/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    SERVICE_QUALITY_THRESHOLDS, RELATIONSHIP_RISK_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)
print("Thresholds and stub data loaded.")

## 4. The Lab 1 tools

In [ ]:
from llama_index.core.workflow import Context


async def get_account_metrics(ctx: Context, account_id: str) -> str:
    """Used by AccountDataAgent. Reads the account's metrics into shared state."""
    data = stub_lookup(account_id)
    state = await ctx.store.get("state")
    if state["account_id"] == "":
        state["account_id"] = account_id
    state["metrics"] = {**data["service_quality"], **data["relationship_risk"]}
    await ctx.store.set("state", state)
    return f"Metrics extracted for {account_id}."


async def get_service_quality_metrics(ctx: Context) -> str:
    """Used by ServiceQualityAgent. Selects the service-quality metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["service_quality_metrics"] = {k: state["metrics"][k] for k in SERVICE_QUALITY_THRESHOLDS}
    state["threshold_service_quality_comments"] = format_threshold_prompt(
        SERVICE_QUALITY_THRESHOLDS, "Service Quality")
    await ctx.store.set("state", state)
    return (f"Service-quality metrics for {state['account_id']}: {state['service_quality_metrics']}\n"
            f"{state['threshold_service_quality_comments']}")


async def get_relationship_risk_metrics(ctx: Context) -> str:
    """Used by RelationshipRiskAgent. Selects the relationship-risk metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["relationship_risk_metrics"] = {k: state["metrics"][k] for k in RELATIONSHIP_RISK_THRESHOLDS}
    state["threshold_relationship_risk_comments"] = format_threshold_prompt(
        RELATIONSHIP_RISK_THRESHOLDS, "Relationship Risk")
    await ctx.store.set("state", state)
    return (f"Relationship-risk metrics for {state['account_id']}: {state['relationship_risk_metrics']}\n"
            f"{state['threshold_relationship_risk_comments']}")


async def get_overall_comments(ctx: Context) -> str:
    """Used by SupervisorAgent. Confirms both scorings are done before the final verdict."""
    state = await ctx.store.get("state")
    if state["threshold_service_quality_comments"] is None:
        return "Service-quality scoring not done."
    if state["threshold_relationship_risk_comments"] is None:
        return "Relationship-risk scoring not done."
    state["overall_comments"] = "Synthesis complete."
    await ctx.store.set("state", state)
    return "Both scorings are complete; write the overall account-health verdict."

## 5. The evaluator's tool: deterministic quality checks

In [ ]:
async def evaluate_assessment(ctx: Context) -> str:
    """Deterministic quality gate on the shared state (no LLM involved)."""
    state = await ctx.store.get("state")
    issues = []
    if state.get("threshold_service_quality_comments") is None:
        issues.append("Missing service-quality analysis.")
    if state.get("threshold_relationship_risk_comments") is None:
        issues.append("Missing relationship-risk analysis.")
    if not state.get("overall_comments"):
        issues.append("Missing overall assessment.")
    if len(state.get("service_quality_metrics", {})) < 4:
        issues.append(f"Only {len(state.get('service_quality_metrics', {}))}/4 service-quality metrics scored.")
    if len(state.get("relationship_risk_metrics", {})) < 4:
        issues.append(f"Only {len(state.get('relationship_risk_metrics', {}))}/4 relationship-risk metrics scored.")
    if issues:
        return "REVISION NEEDED:\n" + "\n".join(f"- {i}" for i in issues)
    return "APPROVED. Assessment complete and consistent."

## 6. Build the agents (now with EvaluatorAgent)

The Supervisor now hands off to the Evaluator; the Evaluator can send it back **at most once** (bounded loop) and writes the final answer on approval.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o", temperature=0)

account_data_agent = FunctionAgent(
    name="AccountDataAgent", description="Collect account metrics.",
    system_prompt="Call get_account_metrics exactly once for the requested account, then hand off to ServiceQualityAgent.",
    llm=llm, tools=[get_account_metrics], can_handoff_to=["ServiceQualityAgent"],
)
service_quality_agent = FunctionAgent(
    name="ServiceQualityAgent", description="Score service quality.",
    system_prompt=("Call get_service_quality_metrics exactly once, then score each service-quality metric from "
                   "1 to 10 against the thresholds, with a data-backed justification. Then hand off to "
                   "RelationshipRiskAgent."),
    llm=llm, tools=[get_service_quality_metrics], can_handoff_to=["RelationshipRiskAgent"],
)
relationship_risk_agent = FunctionAgent(
    name="RelationshipRiskAgent", description="Score relationship risk.",
    system_prompt=("Call get_relationship_risk_metrics exactly once, then score each relationship-risk metric from "
                   "1 to 10 against the thresholds (lower-is-better metrics score high when low). Then hand off "
                   "to SupervisorAgent."),
    llm=llm, tools=[get_relationship_risk_metrics], can_handoff_to=["SupervisorAgent"],
)
supervisor_agent = FunctionAgent(
    name="SupervisorAgent", description="Synthesise the account-health verdict.",
    system_prompt=("Call get_overall_comments exactly once, then write the combined assessment with each metric's "
                   "1-10 score and an overall verdict, and hand off to EvaluatorAgent for review."),
    llm=llm, tools=[get_overall_comments], can_handoff_to=["EvaluatorAgent"],
)
evaluator_agent = FunctionAgent(
    name="EvaluatorAgent", description="Review assessment quality.",
    system_prompt=("You are the quality reviewer. Call evaluate_assessment exactly once. If it returns APPROVED, "
                   "you are the last agent: write the final approved assessment yourself and do not hand off. If "
                   "it returns REVISION NEEDED, hand off to SupervisorAgent with the specific issues, but allow at "
                   "most one revision round before approving."),
    llm=llm, tools=[evaluate_assessment], can_handoff_to=["SupervisorAgent"],
)

## 7. Run the extended workflow

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult

workflow = AgentWorkflow(
    agents=[account_data_agent, service_quality_agent, relationship_risk_agent, supervisor_agent, evaluator_agent],
    root_agent="AccountDataAgent",
    initial_state={
        "metrics": {},
        "account_id": "",
        "service_quality_metrics": {},
        "relationship_risk_metrics": {},
        "threshold_service_quality_comments": None,
        "threshold_relationship_risk_comments": None,
        "overall_comments": None,
    },
)

# Five agents plus a possible revision round need headroom over the default 20 iterations.
handler = workflow.run(user_msg="Assess the account health of AC-RETAIL.", max_iterations=60)
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult):
        print(f"[{event.tool_name}] -> {str(event.tool_output)[:90]}")
result = await handler
print("\n\nFinal:", result)

## 8. Results

**What to notice**
* **Cheap deterministic checks beat LLM-as-judge** for yes/no completeness questions.
* **Incremental design**: you bolted a quality gate onto a working pipeline, the way real systems grow.
* **Bounded loops only**: the revision round is capped; production systems enforce the cap in code too.